In [0]:
# 1. ENVIRONMENT SETUP 
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pyspark.sql.functions as F
import os

# Creating directories inside the /Workspace local filesystem zone
os.makedirs("/Workspace/mock_stream_calls", exist_ok=True)
os.makedirs("/Workspace/checkpoints/bronze_calls", exist_ok=True)

# Defining paths
RAW_CALLS_STREAM_PATH = "/Workspace/mock_stream_calls/"
CHECKPOINT_CALLS = "/Workspace/checkpoints/bronze_calls/"

print("Global imports, schema definitions, and workspace paths successfully initialized.")

Global imports, schema definitions, and workspace paths successfully initialized.


In [0]:
# 2. BRONZE CALLS INGESTION (STREAMING PREPARATION)

# Setting up the streaming reader to capture call data strings dynamically
bronze_calls_stream = (
    spark.readStream
    .format("text") 
    .load(RAW_CALLS_STREAM_PATH)
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

# Starting background stream processing directly to a Delta Table using availableNow
calls_query = (
    bronze_calls_stream.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT_CALLS)
    .toTable("bronze_calls")
)
calls_query.awaitTermination()
# Reading directly from the catalog table i manually uploaded via the UI
bronze_customers_df = (
    spark.read
    .table("workspace.default.customer_cdc_data_final")
    .withColumn("source_file", F.lit("customer_cdc_data_final.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

# Saving customer records to our formal bronze Delta layer
bronze_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_customers")

print("Bronze layer tables ('bronze_calls' & 'bronze_customers') populated successfully.")

Bronze layer tables ('bronze_calls' & 'bronze_customers') populated successfully.


In [0]:
# 3. STREAMING SIMULATOR (MOCK DATA FILE GENERATION)
import json
import os

# Creating mock call records
mock_calls = [
    {"call_id": "C101", "customer_id": 4, "epoch_time": 1775464680, "conversation": {"sentiment": "Positive", "duration_secs": 120}, "metadata": {"agent_id": "A01"}},
    {"call_id": "C102", "customer_id": 28, "epoch_time": 1775464740, "conversation": {"sentiment": "Negative", "duration_secs": 340}, "metadata": {"agent_id": "A02"}},
    {"call_id": "C101", "customer_id": 4, "epoch_time": 1775464680, "conversation": {"sentiment": "Positive", "duration_secs": 120}, "metadata": {"agent_id": "A01"}}, # Intentional duplicate row
    {"call_id": "C103", "customer_id": 70, "epoch_time": 1775465100, "conversation": {"sentiment": "Negative", "duration_secs": 450}, "metadata": {"agent_id": "A03"}}
]

# Converting to JSON lines format
json_lines = "\n".join([json.dumps(record) for record in mock_calls])

# wrting the file directly to the streaming landing folder
with open("/Workspace/mock_stream_calls/calls_batch_1.json", "w") as f:
    f.write(json_lines)

# Re-triggering the bronze background query batch to ingest this new file instantly
calls_query_retrigger = (
    spark.readStream
    .format("text") 
    .load(RAW_CALLS_STREAM_PATH)
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT_CALLS)
    .toTable("bronze_calls")
)

calls_query_retrigger.awaitTermination()

print("Mock streaming files dropped and processed into raw Bronze tables.")    

Mock streaming files dropped and processed into raw Bronze tables.


In [0]:

# 4. SILVER LAYER PROCESSING (CLEANSING & SCD TYPE 2 SCHEMAS)

from pyspark.sql.types import LongType, StructType, StructField

# CUSTOMER CDC LOG TRACKING TO SCD TYPE 2 HISTORY
raw_bronze_customers = spark.read.table("bronze_customers")

cleaned_customers_df = raw_bronze_customers.filter(
    F.col("customer_id").isNotNull() & (F.col("operation") != "")
)

customer_window = Window.partitionBy("customer_id").orderBy("update_ts")

scd2_customers_df = (
    cleaned_customers_df
    .withColumn("start_date", F.to_timestamp(F.col("update_ts")))
    .withColumn("next_start_date", F.lead("start_date", 1).over(customer_window))
    .withColumn("end_date", F.when(F.col("next_start_date").isNotNull(), F.col("next_start_date"))
                           .otherwise(F.to_timestamp(F.lit("9999-12-31 23:59:59"))))
    .withColumn("is_current", F.when(F.col("next_start_date").isNull() & (F.col("operation") != "DELETE"), 1)
                             .otherwise(0))
    .drop("next_start_date")
)

scd2_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_customers_scd2")

# CALL STREAM TRANSFORMATION & NESTED JSON PARSING
json_schema = StructType([
    StructField("call_id", StringType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("epoch_time", LongType(), True),
    StructField("conversation", StructType([
        StructField("sentiment", StringType(), True),
        StructField("duration_secs", IntegerType(), True)
    ]), True),
    StructField("metadata", StructType([
        StructField("agent_id", StringType(), True)
    ]), True)
])

# Reading raw stream from bronze layer
raw_stream_df = spark.readStream.table("bronze_calls")

# Transforming, flattenning, watermarking, and deduplicating
silver_calls_stream = (
    raw_stream_df
    .withColumn("parsed_data", F.from_json(F.col("value"), json_schema))
    .select(
        F.col("parsed_data.call_id").alias("call_id"),
        F.col("parsed_data.customer_id").alias("customer_id"),
        F.to_timestamp(F.col("parsed_data.epoch_time")).alias("event_ts"),
        F.col("parsed_data.conversation.sentiment").alias("sentiment"),
        F.col("parsed_data.conversation.duration_secs").alias("duration_secs"),
        F.col("parsed_data.metadata.agent_id").alias("agent_id"),
        F.col("ingestion_timestamp")
    )
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicates(["call_id", "event_ts"])
)

# Write stream to Delta table
silver_calls_query = (
    silver_calls_stream.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/silver_calls_clean")
    .toTable("workspace.default.silver_calls_clean")
)
silver_calls_query.awaitTermination()
print("Silver datasets ('silver_customers_scd2' & 'silver_calls_clean') fully processed.")

Silver datasets ('silver_customers_scd2' & 'silver_calls_clean') fully processed.


In [0]:
# 5. GOLD LAYER AGGREGATIONS (BUSINESS METRICS & PIT JOINS)

# Load both fully cleaned Silver tables from Unity Catalog
silver_calls = spark.read.table("workspace.default.silver_calls_clean")
silver_customers = spark.read.table("workspace.default.silver_customers_scd2")

#  Perform Point-In-Time Join matching call logs to active customer history windows
enriched_calls_df = silver_calls.join(
    silver_customers,
    (silver_calls.customer_id == silver_customers.customer_id) & 
    (silver_calls.event_ts >= silver_customers.start_date) & 
    (silver_calls.event_ts <= silver_customers.end_date),
    "inner"
).select(
    silver_calls["call_id"],
    silver_calls["customer_id"],
    silver_calls["event_ts"],
    silver_calls["sentiment"],
    silver_calls["duration_secs"],
    silver_customers["city"],
    silver_customers["subscription_type"],
    F.to_date(silver_calls["event_ts"]).alias("call_date")
)

# KPI 1: Daily Sentiment Distribution
gold_sentiment_daily = enriched_calls_df.groupBy("call_date", "sentiment").agg(F.count("call_id").alias("total_calls"))
gold_sentiment_daily.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_sentiment_daily")

# KPI 2: Daily Call Volume
gold_call_volume_daily = enriched_calls_df.groupBy("call_date").agg(F.count("call_id").alias("total_calls"), F.round(F.avg("duration_secs"), 2).alias("avg_duration_secs"))
gold_call_volume_daily.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_call_volume_daily")

# KPI 3: High-Risk Customers
gold_high_risk_customers = enriched_calls_df.filter(F.col("sentiment") == "Negative").groupBy("customer_id").agg(F.count("call_id").alias("negative_call_count")).filter(F.col("negative_call_count") >= 1)
gold_high_risk_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_high_risk_customers")

# KPI 4: Region-wise Sentiment
gold_region_sentiment = enriched_calls_df.groupBy("city", "sentiment").agg(F.count("call_id").alias("call_count"))
gold_region_sentiment.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_region_sentiment")

# KPI 5: Subscription-based Sentiment
gold_subscription_sentiment = enriched_calls_df.groupBy("subscription_type", "sentiment").agg(F.count("call_id").alias("call_count"))
gold_subscription_sentiment.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_subscription_sentiment")

print("Short-term KPI calculations complete!")
print("SUCCESS! : End-to-End Medallion Pipeline executed. All 5 Gold Layer KPIs are fully populated!")

# Show sample output row validations
print("\n--- Displaying Sample Verification Row Results (KPI 4) ---")
display(spark.sql("SELECT * FROM workspace.default.gold_region_sentiment LIMIT 5"))

Short-term KPI calculations complete!
SUCCESS! : End-to-End Medallion Pipeline executed. All 5 Gold Layer KPIs are fully populated!

--- Displaying Sample Verification Row Results (KPI 4) ---


city,sentiment,call_count
Delhi,Positive,1
Bangalore,Negative,2
Mumbai,Positive,1
